In [ ]:
import folium
import pandas as pd
import geopandas as gpd
import requests

In [ ]:
# Load Irish county boundaries from Tailte Eireann via data.gov.ie
# Source: https://data.gov.ie/dataset/counties-national-statutory-boundaries-20191

counties = gpd.read_file("data/counties.geojson")

In [ ]:
# Check the data loaded correctly
print(counties.head()) #Print the first few rows of counties.geojson variable
print(counties.columns) #Print column titles from counties.geojson variable

In [ ]:
# Load CSO afforestation data by county
# Source: Central Statistics Office Ireland AFA01 Afforestation Area 2023
# https://www.cso.ie

forestry = pd.read_csv("data/forestry.csv") #Create DataFrame 'forestry'

# Check the data loaded correctly
print(forestry.head())
print(forestry.columns)

In [ ]:
# heck unique county names in forestry data
print(forestry['County'].unique())
print(len(forestry['County'].unique()))

In [ ]:
print(forestry['Species'].unique())

In [ ]:
print(forestry['Forest Owner'].unique())

In [ ]:
def prepare_forestry_data(df, species, owner='Total Afforestation'):
 
    # Remove Ireland row
    df = df[df['County'] != 'Ireland']
    
    # Filter counties by species and owner
    df = df[(df['Species'] == species) & 
            (df['Forest Owner'] == owner)]
    
    # Clean county names to match GeoJSON format
    df = df.copy()
    df['County'] = df['County'].str.replace('Co. ', '').str.upper()
    
    # Replace NaN values with 0
    df['VALUE'] = df['VALUE'].fillna(0)
    
    return df

In [ ]:
# Prep data for all three map layers
total = prepare_forestry_data(forestry, 'Total Afforestation')
broadleaf = prepare_forestry_data(forestry, 'Broadleaf')
conifer = prepare_forestry_data(forestry, 'Conifer')

print("Total rows:", len(total))
print("Broadleaf rows:", len(broadleaf))
print("Conifer rows:", len(conifer))

In [ ]:
#Check county names have been standardized
print(total['County'].unique())

In [ ]:
# Check CRS of counties data
print(counties.crs)

In [ ]:
# Convert from Irish Transverse Mercator
# to WGS84
counties = counties.to_crs(epsg=4326)

# Verify the conversion
print(counties.crs)

In [ ]:
# Visualize county borders
m = counties.explore('COUNTY', cmap='viridis')
m #show the map

In [ ]:
def merge_forestry_data(counties_gdf, forestry_df, species):

    #merge boundaries with forestry data on county name
    merged = counties_gdf.merge(forestry_df, left_on='COUNTY', right_on='County')
    
    #Drop unneccesary columns
    merged = merged.drop(columns=['Statistic Label', 'Year', 
                                  'Species', 'Forest Owner', 
                                  'UNIT', 'County'])
    return merged

In [ ]:

merged_total = merge_forestry_data(counties, total, 'Total Afforestation')
merged_broadleaf = merge_forestry_data(counties, broadleaf, 'Broadleaf')
merged_conifer = merge_forestry_data(counties, conifer, 'Conifer')

print("Total rows:", len(merged_total))
print("Broadleaf rows:", len(merged_broadleaf))
print("Conifer rows:", len(merged_conifer))

In [ ]:
#Total afforestation map
m = merged_total.explore('VALUE',
                         cmap='YlGn',
                         style_kwds={'fillOpacity': 0.7},
                         legend=True,
                         legend_kwds={'caption': 'Total Afforestation 2023 (Hectares)'},
                         name='Total Afforestation',
                         tiles='CartoDB positron'
                         )
m

In [ ]:
#broadleaf layer to existing map
merged_broadleaf.explore('VALUE',
                          m=m,
                          cmap='YlGn',
                          style_kwds={'fillOpacity': 0.7},
                          legend=True,
                          legend_kwds={'caption': 'Broadleaf Afforestation 2023 (Hectares)'},
                          name='Broadleaf'
                          )

#coniffer layer to existing map
merged_conifer.explore('VALUE',
                        m=m,
                        cmap='YlGn',
                        style_kwds={'fillOpacity': 0.7},
                        legend=True,
                        legend_kwds={'caption': 'Conifer Afforestation 2023 (Hectares)'},
                        name='Conifer'
                        )

In [ ]:
# layer control button
folium.LayerControl().add_to(m)

m

In [ ]:
#Save map to html
m.save('ireland_woodlandmap.html')